# Quick DITL Example

## Introduction

This notebook demonstrates a basic Day-In-The-Life (DITL) simulation using COASTSim. We'll set up a spacecraft configuration, compute its orbit, add observation targets, run the simulation, and visualize the results.

## Step 1: Import Libraries

We start by importing the necessary libraries for configuration, ephemeris computation, visualization, and simulation.

In [ ]:
%matplotlib ipympl
from datetime import datetime

import numpy as np
from rust_ephem import TLEEphemeris

from conops.config.config import MissionConfig
from conops.ditl.queue_ditl import QueueDITL
from conops.visualization import plot_ditl_timeline, plot_sky_pointing

## Step 2: Create Spacecraft Configuration

Create a basic spacecraft configuration using MissionConfig. This sets up default parameters for the spacecraft bus, solar panels, payload, etc.

In [ ]:
config = MissionConfig()

## Step 3: Set up Ephemeris

Load the spacecraft's orbital ephemeris from a TLE file. This defines the
spacecraft's position over time for the simulation period. The ephemeris
pre-calculates the position of the spacecraft, along with the positions of the
Sun and Moon, for every time interval from beginning to end, with a interval of
60 seconds. We use the `rust-ephem` library to compute this for speed.

In [ ]:
ephemeris = TLEEphemeris(
    begin=datetime(2025, 12, 1, 0, 0, 0),
    end=datetime(2025, 12, 2, 0, 0, 0),
    step_size=60,
    tle="example.tle",
)

## Step 3b: Example Star Tracker Configuration

Optionally configure multiple star trackers with different boresight orientations
and constraints. There are two distinct constraint tiers:

- **Hard constraints** — absolute health-and-safety keep-outs (e.g. preventing the
  sensor from staring into the Sun or Earth limb). These are **always** enforced
  in every ACS mode regardless of `modes_require_lock`.
- **Soft constraints** — science-quality indicators (e.g. degraded tracking accuracy
  near bright extended sources). These are only enforced in the modes listed in
  `modes_require_lock`.

After creating the ephemeris, attach it to the star tracker subsystem.

In [ ]:
from rust_ephem import EarthLimbConstraint, SunConstraint

from conops.common.enums import ACSMode
from conops.config import (
    Constraint,
    StarTracker,
    StarTrackerConfiguration,
    StarTrackerOrientation,
    create_star_tracker_vector,
)

offvec = create_star_tracker_vector(0, 45, 0)
vec = create_star_tracker_vector(0, -45, 0)


config.spacecraft_bus.star_trackers = StarTrackerConfiguration(
    star_trackers=[
        StarTracker(
            name="ST-Forward 90 off",
            orientation=StarTrackerOrientation(boresight=offvec),
            soft_constraint=Constraint(
                earth_constraint=EarthLimbConstraint(min_angle=40),
                sun_constraint=SunConstraint(min_angle=20),
            ),
            hard_constraint=Constraint(
                sun_constraint=SunConstraint(min_angle=20),
            ),
        ),
        StarTracker(
            name="ST-Forward",
            orientation=StarTrackerOrientation(boresight=vec),
            soft_constraint=Constraint(
                earth_constraint=EarthLimbConstraint(min_angle=40),
                sun_constraint=SunConstraint(min_angle=20),
            ),
            hard_constraint=Constraint(
                sun_constraint=SunConstraint(min_angle=20),
            ),
        ),
        #        StarTracker(
        #            name="ST-Port",
        #            orientation=StarTrackerOrientation(boresight=(0.0, 1.0, 0.0)),
        #            hard_constraint=Constraint(
        #                earth_constraint=EarthLimbConstraint(min_angle=10),
        #                sun_constraint=SunConstraint(min_angle=20),
        #            ),
        #        ),
    ],
    min_functional_trackers=2,
    modes_require_lock=[ACSMode.SCIENCE],
)
config.spacecraft_bus.star_trackers.set_ephem(ephemeris)
vec

## Step 4: Create DITL and Add Observations

Create a DITL (Data in the Life) simulation object and populate it with random
astronomical observations. Each observation has a random sky position, fixed 1ks
exposure time, and a unique observation ID.

When initializing the DITL, we pass it the `MissionConfig` and `Ephemeris`
objects as arguments.

In [ ]:
import tqdm

config.constraint.ignore_roll = True

ditl = QueueDITL(config=config, ephem=ephemeris)

# Add 1000 random observations to the observation queue
for i in tqdm.tqdm(range(1000)):
    ditl.queue.add(
        ra=np.random.uniform(0, 360),
        dec=np.random.uniform(-90, 90),
        exptime=1000,
        obsid=10000 + i,
    )

## Step 5: Run the Simulation

Execute the DITL calculation to simulate the spacecraft operations, including
scheduling observations, managing power, thermal constraints, and
communications.

QueueDITL uses a queue scheduler, so it schedules targets on the fly during the
DITL. The end result is much the same as if we had pre-created a schedule.

In [ ]:
%%time
ditl.calc()

## Step 6: Visualize Results

Plot the results of the simulation, showing the sky pointing and timeline of
spacecraft operations. 

Figure 1 is an interactive map that shows the pointing of the spacecraft over
time along with various obserinv constraints that are defined by the default
configuration, most prominently the Earth (+ some limb avoidance angle), the
Moon and the Sun.

In [ ]:
import matplotlib.pyplot as plt

_ = plot_sky_pointing(ditl, figsize=(10, 5))
plt.show()

## Figure 2b: Rotatable Globe View

The same data on an interactive 3D globe — click-drag to rotate, scroll to zoom.
The time slider (and Play button) step through the simulation just like the flat map above.


In [ ]:
from conops.visualization import plot_sky_pointing_globe

fig = plot_sky_pointing_globe(ditl)
fig.show()

In [ ]:
_ = plot_ditl_timeline(ditl, figsize=(8, 6))

In [ ]:
from conops.visualization import (
    plot_acs_mode_distribution,
    plot_data_management_telemetry,
)

plot_acs_mode_distribution(ditl)
plot_data_management_telemetry(ditl)

Figure 3 shows the breakdown of time spent in each ACS mode (science, slewing, SAA passage, ground-station pass, charging) as a pie chart, and Figure 4 plots the data management telemetry (buffer fill level and downlink events) across the simulated day.

## Step 7: Inspect the Event Log

The DITL log records every discrete event that occurred during the simulation — mode transitions, target acquisitions, SAA entries/exits, ground-station passes, and fault responses. Printing it gives a timestamped audit trail of spacecraft activity.

In [ ]:
for ev in ditl.log.events:
    print(ev)

## Step 8: Fault Management Timeline

Plot the fault management timeline to see when — and for how long — the spacecraft entered emergency charging or safe-mode, and how quickly autonomous recovery was triggered.

In [ ]:
from conops.visualization.fault_management import plot_fault_management_timeline

plot_fault_management_timeline(
    ditl.config.fault_management,
    t0=ditl.begin.timestamp(),
    t_end=ditl.end.timestamp(),
)

## Step 9: Save the Plan to Disk

Serialise the final observation plan to a JSON file.  Passing `"."` (the current directory) lets COASTSim auto-generate the filename — `plan_<start>_<end>_v<N>.json` — and auto-increment the version counter if a plan for the same time window already exists.  The returned `Path` shows exactly where the file was written.

In [ ]:
ditl.plan.save(".")

In [ ]:
config.constraint.constraint.model_dump()

In [ ]:
plt.figure()
plt.plot(ditl.roll)
plt.show()

In [ ]:
ditl.plot()

In [ ]:
set(ditl.telemetry.housekeeping.star_tracker_functional_count)